# 02 · Silver Transformation – Cleanse, Standardise, Conform
**Layer:** Silver · Reads `bronze_*` → writes `silver_*`

What happens here:
1. **Type casting** – strings → dates, integers, decimals
2. **Standardisation** – trimming, casing, code mapping (e.g. `M`, `male`, ` Male ` → `Male`)
3. **De-duplication** – latest record per business key
4. **Conformance** – foreign keys aligned with reference data
5. **Derived attributes** – age, tenure, full name
6. **Quarantine** – rows failing critical rules go to `quarantine_*`, not silently dropped

In [ ]:
from pyspark.sql import functions as F, Window as W

def latest(df, keys, order_col="_ingested_at"):
    w = W.partitionBy(*keys).orderBy(F.col(order_col).desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def drop_audit(df):
    return df.drop("_source_file", "_batch_id")

def save(df, name):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(f"✔ {name:<32} {spark.table(name).count():>8,} rows")

## Reference data

In [ ]:
save(drop_audit(spark.table("bronze_hris_departments"))
     .withColumn("department_id", F.upper(F.trim("department_id"))), "silver_department")

save(drop_audit(spark.table("bronze_hris_locations")), "silver_location")

save(drop_audit(spark.table("bronze_hris_grades"))
     .withColumn("grade_level", F.col("grade_level").cast("int")), "silver_grade")

save(drop_audit(spark.table("bronze_lms_courses"))
     .withColumn("duration_hours", F.col("duration_hours").cast("int")), "silver_course")

## Employee master (core of the H2R model)

In [ ]:
gender_map = (F.when(F.lower(F.trim("gender")).isin("m", "male"), "Male")
                .when(F.lower(F.trim("gender")).isin("f", "female"), "Female")
                .when(F.lower(F.trim("gender")).isin("non-binary", "nb"), "Non-binary")
                .otherwise("Not Disclosed"))

emp = (spark.table("bronze_hris_employees")
       .withColumn("employee_id",      F.upper(F.trim("employee_id")))
       .withColumn("department_id",    F.upper(F.trim("department_id")))
       .withColumn("gender",           gender_map)
       .withColumn("hire_date",        F.to_date("hire_date"))
       .withColumn("termination_date", F.to_date("termination_date"))
       .withColumn("date_of_birth",    F.to_date("date_of_birth"))
       .withColumn("full_name",        F.concat_ws(" ", F.initcap("first_name"), F.initcap("last_name")))
       .withColumn("as_of_date",       F.coalesce("termination_date", F.current_date()))
       .withColumn("tenure_years",     F.round(F.months_between("as_of_date", "hire_date") / 12, 1))
       .withColumn("age",              F.floor(F.months_between(F.current_date(), "date_of_birth") / 12))
       .withColumn("tenure_band",
            F.when(F.col("tenure_years") < 1, "0-1 Yr").when(F.col("tenure_years") < 3, "1-3 Yrs")
             .when(F.col("tenure_years") < 5, "3-5 Yrs").otherwise("5+ Yrs"))
       .withColumn("age_band",
            F.when(F.col("age") < 25, "<25").when(F.col("age") < 35, "25-34")
             .when(F.col("age") < 45, "35-44").otherwise("45+"))
       .withColumn("is_active", F.col("termination_date").isNull()))

emp = latest(emp, ["employee_id"])

# Critical rules -> quarantine instead of loading bad data
valid_depts = spark.table("silver_department").select("department_id")
bad = (emp.join(valid_depts, "department_id", "left_anti")
          .unionByName(emp.filter("hire_date IS NULL OR (termination_date < hire_date)"))
          .dropDuplicates(["employee_id"]))
bad.withColumn("_reason", F.lit("Invalid department / dates")) \
   .write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("quarantine_employee")

emp_clean = emp.join(bad.select("employee_id"), "employee_id", "left_anti").drop("as_of_date", "_source_file", "_batch_id")
save(emp_clean, "silver_employee")

## Recruitment

In [ ]:
save(drop_audit(spark.table("bronze_ats_requisitions"))
     .withColumn("opened_date", F.to_date("opened_date")), "silver_requisition")

save(latest(drop_audit(spark.table("bronze_ats_candidates")), ["candidate_id"])
     .withColumn("applied_date", F.to_date("applied_date"))
     .withColumn("stage_order", F.expr("""CASE current_stage WHEN 'Applied' THEN 1 WHEN 'Screened' THEN 2
                                           WHEN 'Interviewed' THEN 3 WHEN 'Offered' THEN 4 WHEN 'Hired' THEN 5 END""")),
     "silver_candidate")

## Onboarding, Learning, Leave, Performance, Payroll, Exits

In [ ]:
save(drop_audit(spark.table("bronze_hris_onboarding"))
     .withColumn("onboarding_start", F.to_date("onboarding_start"))
     .withColumn("laptop_issued_days", F.col("laptop_issued_days").cast("int"))
     .withColumn("onboarding_survey_score", F.col("onboarding_survey_score").cast("decimal(3,1)")),
     "silver_onboarding")

save(drop_audit(spark.table("bronze_lms_training_records"))
     .withColumn("enrolled_date", F.to_date("enrolled_date"))
     .withColumn("completed_date", F.to_date("completed_date"))
     .withColumn("assessment_score", F.col("assessment_score").cast("int")),
     "silver_training")

save(drop_audit(spark.table("bronze_tna_leave_records"))
     .withColumn("leave_start_date", F.to_date("leave_start_date"))
     .withColumn("leave_days", F.col("leave_days").cast("int")),
     "silver_leave")

save(drop_audit(spark.table("bronze_perf_reviews"))
     .withColumn("performance_rating", F.col("performance_rating").cast("int"))
     .withColumn("goal_achievement_pct", F.col("goal_achievement_pct").cast("decimal(5,1)"))
     .withColumn("fiscal_year_end", F.to_date(F.concat(F.lit("20"), F.substring("review_cycle", 3, 2), F.lit("-03-31")))),
     "silver_performance")

pay = drop_audit(spark.table("bronze_payroll_monthly")).withColumn("pay_month", F.to_date("pay_month"))
for c in ["base_pay", "benefits", "bonus", "overtime_pay", "deductions"]:
    pay = pay.withColumn(c, F.col(c).cast("decimal(14,2)"))
pay = pay.withColumn("gross_pay", F.col("base_pay") + F.col("benefits") + F.col("bonus") + F.col("overtime_pay")) \
         .withColumn("net_pay", F.col("gross_pay") - F.col("deductions"))
save(pay, "silver_payroll")

save(drop_audit(spark.table("bronze_hris_exits"))
     .withColumn("resignation_date", F.to_date("resignation_date"))
     .withColumn("last_working_date", F.to_date("last_working_date"))
     .withColumn("notice_days", F.datediff("last_working_date", "resignation_date"))
     .withColumn("exit_interview_score", F.col("exit_interview_score").cast("decimal(3,1)")),
     "silver_exit")